# Feature Extraction in Text — Full Pipeline

This notebook runs the complete, hand-built pipeline end to end on real
documents scraped from Hacker News (`hackernews_dataset.csv`, produced by
`scrape_hackernews.py`), falling back to the small curated `demo.csv` corpus
if that file isn't present:

1. Load raw text from CSV
2. Preprocess (lowercase, strip punctuation/numbers, tokenize, remove
   stopwords, stem)
3. POS tagging
4. Syntactic features (POS tags -> parse tree -> subject/verb/object,
   noun/verb/adjective counts, clause and dependency information)
5. Semantic features (rule-based lexical categories, plus distributional
   embeddings via co-occurrence -> PPMI -> truncated SVD / LSA)
6. Morphological analysis
7. Lemmatization (vs. stemming)
8. Vocabulary + Bag of Words / Binary BoW
9. One-hot encoding (tokens and POS tags)
10. TF-IDF + document cosine similarity
11. N-grams
12. A short NLTK-based comparison of the preprocessing/tagging steps

Steps 1-11 use only `preprocessing.py`, `linguistic_features.py`,
`syntactic_features.py`, `semantic_features.py`, and `features.py` in this
project — no scikit-learn, nltk, or spaCy. Step 12 brings in `nltk` purely
to compare against that hand-built pipeline.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from preprocessing import (
    preprocess,
    lowercase,
    remove_punctuation_and_numbers,
    tokenize,
    remove_stopwords,
    stem_word,
    stem_tokens,
)
from linguistic_features import (
    pos_tag,
    pos_tag_word,
    morphological_analysis,
    lemmatize_word,
    lemmatize_tokens,
)
from syntactic_features import *
from semantic_features import (
    semantic_feature_matrix,
    cosine_similarity_matrix,
)
from features import (
    build_vocabulary,
    bag_of_words,
    binary_bow,
    tf_idf,
    n_grams,
    one_hot_encode_tokens,
    one_hot_encode_categories,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 2. Load raw text from CSV

Reads `hackernews_dataset.csv` (real scraped Hacker News stories, one per
row in a `text` column) if it exists, otherwise falls back to the small
curated `demo.csv` corpus.

In [2]:
from pathlib import Path

# csv_path = "hackernews_dataset.csv" if Path("hackernews_dataset.csv").exists() else "demo.csv"
csv_path = "demo.csv"
data = pd.read_csv(csv_path)
corpus = data["text"][:10].tolist()

print(f"Loaded {len(corpus)} documents from {csv_path}\n")
for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Loaded 8 documents from demo.csv

Doc 0: The cat sat on the mat and looked at the dog.
Doc 1: Dogs are running quickly in the park every morning.
Doc 2: The quick brown fox jumps over the lazy dog.
Doc 3: Cats and dogs are popular pets around the world.
Doc 4: She quickly finished reading the interesting book.
Doc 5: The park was full of happy dogs and playful cats.
Doc 6: A curious fox wandered through the quiet forest at night.
Doc 7: Books about dogs and cats are popular with young readers.


## 3. Preprocessing

`preprocess()` runs lowercase -> strip punctuation/numbers -> tokenize ->
remove stopwords -> stem, in one call. We also keep a lighter "raw tokens"
version per document (lowercased and tokenized, but *not* stopword-stripped
or stemmed) for the linguistic analysis steps below, since POS tagging and
morphology need function words and full word forms to work with.


In [3]:
processed_docs = [preprocess(doc) for doc in corpus]
raw_tokens_per_doc = [
    tokenize(remove_punctuation_and_numbers(lowercase(doc))) for doc in corpus
]

print("=== Tokens after full preprocessing (stopwords removed, stemmed) ===")
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}: {tokens}")


=== Tokens after full preprocessing (stopwords removed, stemmed) ===
Doc 0: ['cat', 'sat', 'mat', 'look', 'dog']
Doc 1: ['dog', 'runn', 'quick', 'park', 'every', 'morn']
Doc 2: ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']
Doc 3: ['cat', 'dog', 'popular', 'pet', 'around', 'world']
Doc 4: ['quick', 'finish', 'read', 'interest', 'book']
Doc 5: ['park', 'full', 'happy', 'dog', 'playful', 'cat']
Doc 6: ['curiou', 'fox', 'wander', 'quiet', 'forest', 'night']
Doc 7: ['book', 'dog', 'cat', 'popular', 'young', 'reader']


## 4. POS Tagging

Rule-based tagging (closed-class lexicon + suffix rules) on the raw tokens
of every document, shown as one combined table.


In [4]:
pos_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    for word, tag in pos_tag(tokens):
        pos_rows.append({"doc": doc_index, "word": word, "pos_tag": tag})

pos_df = pd.DataFrame(pos_rows)
pos_df


,doc,word,pos_tag
0,0,the,DET
1,0,cat,NOUN
2,0,sat,VERB
3,0,on,PREP
4,0,the,DET
...,...,...,...
70,7,are,AUX
71,7,popular,ADJ
72,7,with,PREP
73,7,young,ADJ


## 5. Syntactic Features

Full pipeline: tokens -> POS tags -> a small hand-written constituency
parser -> a parse tree, from which we extract:

- **subject / verb / object** — read off the tree's first clause instead of
  guessed from raw token order
- **noun / verb / adjective counts**
- **clause count** — how many `S` (clause) nodes the parser found
- **dependency relations** (`det`, `amod`, `nsubj`, `dobj`, `pobj`) — simple
  head-dependent pairs read off the tree, approximating a dependency parse

In [5]:
print("=== Parse Tree (Doc 0) ===")
parse_tree_doc0 = build_parse_tree(pos_tag(raw_tokens_per_doc[7]))
print(render_tree(parse_tree_doc0))

print("\n=== Dependencies (Doc 0) ===")
for dependency in extract_dependencies(parse_tree_doc0):
    print(f"{dependency['relation']:6s} {dependency['head']} -> {dependency['dependent']}")

syntactic_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    tagged = pos_tag(tokens)
    doc_syntax = syntactic_analysis(tagged).drop(
        columns=["dependencies"],
        errors="ignore",
    )
    doc_syntax.insert(0, "doc", doc_index)
    syntactic_frames.append(doc_syntax)

syntactic_df = pd.concat(syntactic_frames, ignore_index=True)
syntactic_df

=== Parse Tree (Doc 0) ===
ROOT
  NOUN: 'books'
  PREP: 'about'
  NOUN: 'dogs'
  CONJ: 'and'
  NOUN: 'cats'
  AUX: 'are'
  ADJ: 'popular'
  PREP: 'with'
  ADJ: 'young'
  NOUN: 'readers'

=== Dependencies (Doc 0) ===


,doc,subject,verb,object,token_count,noun_count,verb_count,adj_count,clause_count,has_subject_verb_object,dependency_count
0,0,cat,sat,mat,11,3,2,0,2,True,6
1,1,dogs,are,None,9,3,2,0,1,False,1
2,2,fox,jumps,dog,9,2,1,3,1,True,7
3,3,None,None,None,9,4,1,1,0,False,0
4,4,she,finished,None,7,1,2,1,2,False,4
5,5,None,None,None,10,3,1,3,0,False,0
6,6,fox,wandered,forest,10,3,1,2,1,True,7
7,7,None,None,None,10,4,1,2,0,False,0


## 6. Semantic Features

Counts of tokens per hand-written lexical category, drawn from
`SEMANTIC_LEXICON`. The categories cover the vocabulary of *both* of this
project's corpora: `action`, `animal`, `place`, `object`, and `descriptive`
(the animal/nature sentences in `demo.csv`), plus `technology`, `ai`,
`business`, and `science` (the tech news titles/text in
`hackernews_dataset.csv`). `positive_description` and
`negative_description` are shared across both.


In [6]:
semantic_df = semantic_feature_matrix(raw_tokens_per_doc)
semantic_df

,action,animal,place,object,technology,ai,business,science,positive_description,negative_description,descriptive
0,2,2,0,1,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,0,1
2,1,2,0,0,0,0,0,0,0,0,3
3,0,2,1,0,0,0,0,0,1,0,0
4,2,0,0,1,0,0,0,0,1,0,1
5,0,2,1,0,0,0,0,0,2,0,1
6,1,1,1,0,0,0,0,0,0,0,2
7,0,2,0,0,0,0,0,0,1,0,1


## 7. Morphological Analysis

Per-word shape and inflection features (length, vowel/consonant counts,
prefix/suffix, plural/gerund/past-tense/comparative/superlative flags) for
every document, combined into one table.

In [7]:
morph_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    doc_morph = morphological_analysis(tokens)
    doc_morph.insert(0, "doc", doc_index)
    morph_frames.append(doc_morph)

morph_df = pd.concat(morph_frames, ignore_index=True)
morph_df


,doc,word,length,num_vowels,num_consonants,prefix3,suffix3,is_capitalized,is_plural,is_gerund,is_past_tense,is_comparative,is_superlative
0,0,the,3,1,2,the,the,False,False,False,False,False,False
1,0,cat,3,1,2,cat,cat,False,False,False,False,False,False
2,0,sat,3,1,2,sat,sat,False,False,False,True,False,False
3,0,on,2,1,1,on,on,False,False,False,False,False,False
4,0,the,3,1,2,the,the,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,7,are,3,2,1,are,are,False,False,False,False,False,False
71,7,popular,7,3,4,pop,lar,False,False,False,False,False,False
72,7,with,4,1,3,wit,ith,False,False,False,False,False,False
73,7,young,5,2,3,you,ung,False,False,False,False,False,False


## 8. Lemmatization vs. Stemming

Both reduce a word to a base form, but the stemmer just chops suffixes
(sometimes producing fragments that aren't real words), while the
lemmatizer aims to return an actual dictionary word. Comparison shown on
stopword-free tokens from every document.

In [8]:
lemma_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    filtered = remove_stopwords(tokens)
    for word in filtered:
        lemma_rows.append({
            "doc": doc_index,
            "word": word,
            "stem": stem_word(word),
            "lemma": lemmatize_word(word),
        })

lemma_df = pd.DataFrame(lemma_rows)
lemma_df


,doc,word,stem,lemma
0,0,cat,cat,cat
1,0,sat,sat,sat
2,0,mat,mat,mat
3,0,looked,look,look
4,0,dog,dog,dog
5,1,dogs,dog,dog
6,1,running,runn,run
7,1,quickly,quick,quick
8,1,park,park,park
9,1,every,every,every


## 9. Vocabulary

Built from the fully preprocessed (stopword-free, stemmed) tokens.

In [9]:
vocab = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")
vocab


Vocabulary size: 32


['around',
 'book',
 'brown',
 'cat',
 'curiou',
 'dog',
 'every',
 'finish',
 'forest',
 'fox',
 'full',
 'happy',
 'interest',
 'jump',
 'lazy',
 'look',
 'mat',
 'morn',
 'night',
 'park',
 'pet',
 'playful',
 'popular',
 'quick',
 'quiet',
 'read',
 'reader',
 'runn',
 'sat',
 'wander',
 'world',
 'young']

## 10. Bag of Words

In [10]:
bow_df = bag_of_words(processed_docs, vocab)
bow_df


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0
2,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
5,0,0,0,1,0,1,0,0,0,0,1,1,0,0,0,...,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
7,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1


## 11. Binary Bag of Words

In [11]:
binary_df = binary_bow(processed_docs, vocab)
binary_df


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0
2,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
5,0,0,0,1,0,1,0,0,0,0,1,1,0,0,0,...,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
7,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1


## 12. One-Hot Encoding

Two flavors: one-hot per token *position* in a document (preserves order,
unlike bag-of-words), and a generic one-hot encoding of category labels
(here, the POS tags of Doc 0).

In [12]:
print("=== One-hot encoding of tokens (Doc 0) ===")
one_hot_tokens_df = one_hot_encode_tokens(processed_docs[0], vocab)
one_hot_tokens_df


=== One-hot encoding of tokens (Doc 0) ===


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
print("=== One-hot encoding of POS tags (Doc 0) ===")
pos_tags_doc0 = [tag for _, tag in pos_tag(raw_tokens_per_doc[0])]
one_hot_pos_df = one_hot_encode_categories(pos_tags_doc0)
one_hot_pos_df


=== One-hot encoding of POS tags (Doc 0) ===


,CONJ,DET,NOUN,PREP,VERB
0,0,1,0,0,0
1,0,0,1,0,0
2,0,0,0,0,1
3,0,0,0,1,0
4,0,1,0,0,0
5,0,0,1,0,0
6,1,0,0,0,0
7,0,0,0,0,1
8,0,0,0,1,0
9,0,1,0,0,0


## 13. TF-IDF

In [14]:
tfidf_df = tf_idf(processed_docs, vocab)
tfidf_df.round(3)


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0.000,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.288,2.079,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,2.079,0.000,1.386,0.000,0.000,0.000,0.981,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000
2,0.000,0.000,2.079,0.000,0.000,0.288,0.000,0.000,0.000,1.386,0.000,0.000,0.000,2.079,2.079,...,0.000,0.000,0.000,0.000,0.000,0.000,0.981,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,2.079,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,2.079,0.000,1.386,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.079,0.000
4,0.000,1.386,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.981,0.000,2.079,0.000,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,2.079,2.079,0.000,0.000,0.000,...,0.000,0.000,1.386,0.000,2.079,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,2.079,1.386,0.000,0.000,0.000,0.000,0.000,...,0.000,2.079,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079,0.000,0.000
7,0.000,1.386,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,1.386,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079


### Document Similarity (TF-IDF Cosine)

Pairwise cosine similarity between documents' TF-IDF vectors — lexical
overlap, not deep semantic meaning.

In [15]:
cosine_similarity_matrix(tfidf_df).round(3)

,doc_0,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7
doc_0,1.000,0.006,0.006,0.039,0.000,0.039,0.000,0.042
doc_1,0.006,1.000,0.066,0.005,0.061,0.128,0.000,0.006
doc_2,0.006,0.066,1.000,0.005,0.061,0.005,0.099,0.006
doc_3,0.039,0.005,0.005,1.000,0.000,0.036,0.000,0.175
doc_4,0.000,0.061,0.061,0.000,1.000,0.000,0.000,0.134
doc_5,0.039,0.128,0.005,0.036,0.000,1.000,0.000,0.040
doc_6,0.000,0.000,0.099,0.000,0.000,0.000,1.000,0.000
doc_7,0.042,0.006,0.006,0.175,0.134,0.040,0.000,1.000


## 14. N-grams

Bigrams and trigrams for every document, built from the fully preprocessed
tokens.

In [16]:
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}")
    print("  Bigrams: ", n_grams(tokens, 2))
    print("  Trigrams:", n_grams(tokens, 3))


Doc 0
  Bigrams:  [('cat', 'sat'), ('sat', 'mat'), ('mat', 'look'), ('look', 'dog')]
  Trigrams: [('cat', 'sat', 'mat'), ('sat', 'mat', 'look'), ('mat', 'look', 'dog')]
Doc 1
  Bigrams:  [('dog', 'runn'), ('runn', 'quick'), ('quick', 'park'), ('park', 'every'), ('every', 'morn')]
  Trigrams: [('dog', 'runn', 'quick'), ('runn', 'quick', 'park'), ('quick', 'park', 'every'), ('park', 'every', 'morn')]
Doc 2
  Bigrams:  [('quick', 'brown'), ('brown', 'fox'), ('fox', 'jump'), ('jump', 'lazy'), ('lazy', 'dog')]
  Trigrams: [('quick', 'brown', 'fox'), ('brown', 'fox', 'jump'), ('fox', 'jump', 'lazy'), ('jump', 'lazy', 'dog')]
Doc 3
  Bigrams:  [('cat', 'dog'), ('dog', 'popular'), ('popular', 'pet'), ('pet', 'around'), ('around', 'world')]
  Trigrams: [('cat', 'dog', 'popular'), ('dog', 'popular', 'pet'), ('popular', 'pet', 'around'), ('pet', 'around', 'world')]
Doc 4
  Bigrams:  [('quick', 'finish'), ('finish', 'read'), ('read', 'interest'), ('interest', 'book')]
  Trigrams: [('quick', 'finis

## 15. Comparison with NLTK

A short sanity check: run the same corpus through NLTK's tokenizer,
stopword list, Porter stemmer, WordNet lemmatizer, and POS tagger, and
compare the results against our hand-built versions above.

In [17]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag as nltk_pos_tag

for pkg, path in [
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet"),
    ("averaged_perceptron_tagger_eng", "taggers/averaged_perceptron_tagger_eng"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(pkg, quiet=True)

nltk_stop_set = set(nltk_stopwords.words("english"))
porter = PorterStemmer()
wnl = WordNetLemmatizer()

In [18]:
nltk_raw_tokens_per_doc = [word_tokenize(doc.lower()) for doc in corpus]
nltk_processed_docs = [
    [porter.stem(t) for t in tokens if t.isalpha() and t not in nltk_stop_set]
    for tokens in nltk_raw_tokens_per_doc
]

print("=== Tokens after preprocessing: ours vs. NLTK ===")
for i in range(len(corpus)):
    print(f"Doc {i}")
    print("  ours: ", processed_docs[i])
    print("  nltk: ", nltk_processed_docs[i])

=== Tokens after preprocessing: ours vs. NLTK ===
Doc 0
  ours:  ['cat', 'sat', 'mat', 'look', 'dog']
  nltk:  ['cat', 'sat', 'mat', 'look', 'dog']
Doc 1
  ours:  ['dog', 'runn', 'quick', 'park', 'every', 'morn']
  nltk:  ['dog', 'run', 'quickli', 'park', 'everi', 'morn']
Doc 2
  ours:  ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']
  nltk:  ['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']
Doc 3
  ours:  ['cat', 'dog', 'popular', 'pet', 'around', 'world']
  nltk:  ['cat', 'dog', 'popular', 'pet', 'around', 'world']
Doc 4
  ours:  ['quick', 'finish', 'read', 'interest', 'book']
  nltk:  ['quickli', 'finish', 'read', 'interest', 'book']
Doc 5
  ours:  ['park', 'full', 'happy', 'dog', 'playful', 'cat']
  nltk:  ['park', 'full', 'happi', 'dog', 'play', 'cat']
Doc 6
  ours:  ['curiou', 'fox', 'wander', 'quiet', 'forest', 'night']
  nltk:  ['curiou', 'fox', 'wander', 'quiet', 'forest', 'night']
Doc 7
  ours:  ['book', 'dog', 'cat', 'popular', 'young', 'reader']
  nltk:  ['book', 'dog', 'ca

In [19]:
print("=== Stemming vs. lemmatization: ours vs. NLTK (Doc 1) ===")
filtered_doc1 = remove_stopwords(raw_tokens_per_doc[1])
lemma_compare = pd.DataFrame({
    "word": filtered_doc1,
    "our_stem": [stem_word(w) for w in filtered_doc1],
    "nltk_stem": [porter.stem(w) for w in filtered_doc1],
    "our_lemma": [lemmatize_word(w) for w in filtered_doc1],
    "nltk_lemma": [wnl.lemmatize(w) for w in filtered_doc1],
})
lemma_compare

=== Stemming vs. lemmatization: ours vs. NLTK (Doc 1) ===


,word,our_stem,nltk_stem,our_lemma,nltk_lemma
0,dogs,dog,dog,dog,dog
1,running,runn,run,run,running
2,quickly,quick,quickli,quick,quickly
3,park,park,park,park,park
4,every,every,everi,every,every
5,morning,morn,morn,morn,morning


In [ ]:
print("=== POS tags: ours vs. NLTK - Doc 0 ===")
pos_compare = pd.DataFrame({
    "word": raw_tokens_per_doc[0],
    "our_tag": [tag for _, tag in pos_tag(raw_tokens_per_doc[0])],
    "nltk_tag": [tag for _, tag in nltk_pos_tag(raw_tokens_per_doc[0])],
})
pos_compare

=== POS tags: ours (coarse) vs. NLTK (Penn Treebank) - Doc 0 ===


,word,our_tag,nltk_tag
0,the,DET,DT
1,cat,NOUN,NN
2,sat,VERB,VBD
3,on,PREP,IN
4,the,DET,DT
5,mat,NOUN,NN
6,and,CONJ,CC
7,looked,VERB,VBD
8,at,PREP,IN
9,the,DET,DT


In [21]:
nltk_vocab = build_vocabulary(nltk_processed_docs)
print(f"Vocabulary size - ours: {len(vocab)}, NLTK: {len(nltk_vocab)}")
print(f"Shared terms: {len(set(vocab) & set(nltk_vocab))}")
print(f"Only ours: {sorted(set(vocab) - set(nltk_vocab))}")
print(f"Only NLTK: {sorted(set(nltk_vocab) - set(vocab))}")

Vocabulary size - ours: 32, NLTK: 33
Shared terms: 27
Only ours: ['every', 'happy', 'lazy', 'playful', 'runn']
Only NLTK: ['everi', 'happi', 'lazi', 'play', 'quickli', 'run']


### Takeaways

- **Tokenization** matches once both sides drop punctuation/numbers — NLTK's
  tokenizer additionally splits off contractions and punctuation as their
  own tokens, which we don't need since punctuation is stripped up front.
- **Stemming** mostly agrees (`dogs`->`dog`, `park`->`park`), but diverges on
  `-ly`/`-y` endings — Porter turns `quickly`->`quickli` and `happy`->`happi`,
  fragments our simpler suffix list avoids (`quick`, `happy`).
- **Lemmatization** shows the biggest gap: NLTK's `WordNetLemmatizer`
  defaults to treating every word as a noun, so it leaves verb forms like
  `running` unchanged, while our lemmatizer applies suffix rules regardless
  of part of speech — correctly reducing `running`->`run`, but also
  over-stemming `morning`->`morn`.
- **POS tagging**: our tagger only distinguishes a handful of coarse
  categories (NOUN/VERB/DET/PREP/CONJ/...), while NLTK's averaged perceptron
  tagger returns fine-grained, trained Penn Treebank tags (NN, VBD, DT, IN,
  CC, ...).
- **Vocabulary**: sizes are close and most terms are shared — the small
  differences trace directly back to the stemmer divergence above.

Overall, our rule-based pipeline approximates NLTK's behavior well on
regular text, but NLTK's components (trained tagger, dictionary-backed
lemmatizer, more complete stemmer) handle edge cases more robustly.